In [ ]:
!pip install torch torchvision
!pip install torchmetrics
!pip install lightning
!pip install numpy
!pip install wandb

import torch
import torch.nn as nn
import torchmetrics
import pytorch_lightning as pl
from pytorch_lightning import loggers as pl_loggers
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from pathlib import Path
import numpy as np

class EnsureRGB:
    def __call__(self, img):
        return img.convert("RGB")

VALID_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.ppm', '.bmp', '.pgm', '.tif', '.tiff', '.webp'}

def is_valid_file(path):
    return Path(path).suffix.lower() in VALID_EXTENSIONS

data_dir = Path("./data/Training")  # single folder with class subfolders

train_tfms = transforms.Compose([
    EnsureRGB(),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    EnsureRGB(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load full dataset once just to get the split indices
full_ds = datasets.ImageFolder(root=data_dir, transform=train_tfms, is_valid_file=is_valid_file)

train_size = int(0.70 * len(full_ds))
val_size   = int(0.15 * len(full_ds))
test_size  = len(full_ds) - train_size - val_size

torch.manual_seed(42)
indices = torch.randperm(len(full_ds)).tolist()
train_idx = indices[:train_size]
val_idx   = indices[train_size:train_size + val_size]
test_idx  = indices[train_size + val_size:]

# Build separate datasets so val/test use eval transforms (no augmentation)
train_ds_full = datasets.ImageFolder(root=data_dir, transform=train_tfms, is_valid_file=is_valid_file)
eval_ds_full  = datasets.ImageFolder(root=data_dir, transform=eval_tfms, is_valid_file=is_valid_file)

train_ds = Subset(train_ds_full, train_idx)
val_ds   = Subset(eval_ds_full, val_idx)
test_ds  = Subset(eval_ds_full, test_idx)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

b = 16
train_loader = DataLoader(train_ds, batch_size=b, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=b, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=b, shuffle=False)


class LitNetwork(pl.LightningModule):
    def __init__(self, n_classes=4):
        super(LitNetwork, self).__init__()
        
        weights = ResNet50_Weights.DEFAULT
        self.network = resnet50(weights=weights)

        #Adjust the final fully connected layer to match the number of classes
        num_ftrs = self.network.fc.in_features
        self.network.fc = nn.Linear(num_ftrs, n_classes)

        self.loss_func = nn.CrossEntropyLoss()
        self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes, average='micro')
        self.val_acc = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes, average='micro')
        self.test_acc = torchmetrics.Accuracy(task="multiclass", num_classes=n_classes, average='micro')

    def forward(self, x):
        return self.network(x)

    def configure_optimizers(self):
        # Differential learning rates: higher for new fc head, lower for pretrained backbone
        backbone_params = [p for name, p in self.network.named_parameters()
                           if not name.startswith("fc") and p.requires_grad]
        head_params = list(self.network.fc.parameters())

        optimizer = torch.optim.AdamW([
            {"params": backbone_params, "lr": 1e-4},
            {"params": head_params, "lr": 1e-3},
        ], weight_decay=1e-4)

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
          optimizer,
          mode="max",      # because we monitor val_acc
          factor=0.5,
          threshold=1e-3,      # require meaningful improvement
          threshold_mode="rel",
          min_lr=1e-6
      )
        return {
          "optimizer": optimizer,
          "lr_scheduler": {
              "scheduler": scheduler,
              "monitor": "val_acc",
              "interval": "epoch",
              "frequency": 1
          }
      }

    def training_step(self, data, batch_idx):
        features, target = data[0], data[1]
        outs = self.forward(features)
        loss = self.loss_func(outs, target)
        self.train_acc(outs, target)
        self.log("train_loss", loss)
        self.log("train_acc", self.train_acc, prog_bar=True, on_step=False, on_epoch=True)
        return loss
    
    def validation_step(self, val_data, batch_idx):
        features, target = val_data[0], val_data[1]
        outs = self.forward(features)
        self.val_acc(outs,target)
        self.log("val_acc",self.val_acc,prog_bar=True,on_step=False,on_epoch=True)
        return None

    def test_step(self, test_data, batch_idx):
        features, target = test_data[0], test_data[1]
        outs = self.forward(features)
        self.test_acc(outs,target)
        self.log("test_acc",self.test_acc,prog_bar=True,on_step=False,on_epoch=True)
        return None

device = "gpu"

# --- Fine-tuned: all layers train ---
print("=== Training FINE-TUNED (all layers) ===")
finetuned_model = LitNetwork(n_classes=4)
finetuned_logger = pl_loggers.WandbLogger(project="hammer-classification", name="finetuned", log_model="all")
finetuned_checkpoint = pl.callbacks.ModelCheckpoint(monitor='val_acc', save_top_k=1, mode='max')
finetuned_early_stop = pl.callbacks.EarlyStopping(monitor='val_acc', mode='max', patience=10)

finetuned_trainer = pl.Trainer(max_epochs=50, accelerator=device, callbacks=[finetuned_checkpoint, finetuned_early_stop], logger=finetuned_logger)
finetuned_trainer.fit(finetuned_model, train_loader, val_loader)
finetuned_trainer.test(ckpt_path="best", dataloaders=test_loader)

torch.save(finetuned_model.network.state_dict(), "finetuned.pt")
print("Saved finetuned.pt")